# Exercício 1: PINN mínima versus Runge–Kutta (RK45)

Equipe: José Salgado, Renan Brêttas e Heitor Morais.

Este notebook roda **todo** o exercício 1 do repositório, do início ao fim, sem intervenção: baixa o código,
executa `exercicio1.py` e mostra a tabela e as figuras que o relatório usa.

**Como usar no Colab:** menu *Ambiente de execução → Executar tudo* (o Colab pede login com uma conta Google).
Leva poucos minutos em CPU.

In [ ]:
# 1. Baixa o repositório (se ainda não estivermos dentro dele) e entra na pasta.
import os, subprocess, sys

REPO = "jssppedra-png/trabalho-pinn-runge-kutta"
PASTA = REPO.split("/")[1]

if not os.path.exists("exercicio1.py"):
    if not os.path.isdir(PASTA):
        url = f"https://github.com/{REPO}.git"
        try:  # no Colab, usa o token guardado nos Secrets, se houver (necessário se o repositório for privado)
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        r = subprocess.run(["git", "clone", "-q", url, PASTA], capture_output=True, text=True)
        if r.returncode != 0:  # não mostra a mensagem do git, que poderia conter o token
            raise RuntimeError("Não consegui baixar o repositório. Se ele for privado, confira o GITHUB_TOKEN nos Secrets.")
    os.chdir(PASTA)

print("Pasta de trabalho:", os.getcwd())

## 2. Versões das ferramentas

Os números do relatório foram gerados com as versões de `requirements.txt` (Python 3.14, CPU). O Colab traz versões
próprias. Os erros do RK45 se repetem; os da PINN podem mudar bastante (num teste no Colab, o do problema (ii) ficou
cerca de 6 vezes maior), porque o resultado do treino depende da versão do PyTorch e do processador. A conclusão não
muda: a PINN continua muito atrás do RK45. Os tempos dependem da máquina.
Para usar exatamente as mesmas versões, mude `INSTALAR_VERSOES_EXATAS` para `True` (demora mais).

In [ ]:
INSTALAR_VERSOES_EXATAS = False
if INSTALAR_VERSOES_EXATAS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import platform, numpy, scipy, torch, matplotlib
print("Python    ", platform.python_version())
print("numpy     ", numpy.__version__)
print("scipy     ", scipy.__version__)
print("torch     ", torch.__version__)
print("matplotlib", matplotlib.__version__)

## 3. Roda o exercício inteiro

Um comando só: treina as PINNs, roda o RK45 e grava tabelas, figuras e números em `resultados/`.
A semente aleatória é fixada uma única vez, no início do script.

In [ ]:
!{sys.executable} exercicio1.py

## 4. Resultados

Erro máximo numa grade de 1000 pontos (mesma régua para os dois métodos), tempo e número de avaliações de f.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("resultados/tabela.md", encoding="utf-8").read()))

In [ ]:
FIGURAS = [
    ("fig2_custo_precisao.png", "Custo × precisão: curva do RK45 (várias tolerâncias) e pontos das PINNs. Abaixo e à esquerda é melhor."),
    ("fig1_solucoes.png", "Resposta certa, passos do RK45 e PINN com 5 mil iterações."),
    ("fig3_remedios_rigido.png", "Problema (i): a falha da PINN e dois remédios para a condição inicial."),
    ("fig4_fase_coelhos_raposas.png", "Problema (iii) no plano de fase: com 5 mil iterações a PINN espirala para o equilíbrio."),
]
for arquivo, legenda in FIGURAS:
    display(Markdown(f"**{legenda}**"))
    display(Image(filename=os.path.join("resultados", arquivo), width=900))

## 5. (Opcional) Um problema de cada vez

Cada método também roda sozinho. Os nomes dos problemas são `rigido` (y′ = −50y), `oscilante` (y′ = cos(20t))
e `coelhos_raposas` (Lotka–Volterra).

In [ ]:
!{sys.executable} rk.py oscilante
!{sys.executable} pinn.py oscilante